# 03. Model Training and Benchmarking Pipeline

Comparing Scikit-Learn regression pipelines across Random Forest and XGBoost estimators, running k-fold cross validation, and exporting the production model artifact.

In [ ]:
import sys
import os
import joblib
import pandas as pd
import numpy as np
sys.path.append('..')

from src.models.train import build_pipeline, prepare_data
from src.models.evaluate import evaluate_model

### 1. Prepare Training and Evaluation Partitions

In [ ]:
data_file = '../data/raw/laptop_price - dataset.csv'
if os.path.exists(data_file):
    X_train, y_train, X_test, y_test = prepare_data(data_file)
    print(f'Training observations: {len(X_train)}, Testing observations: {len(X_test)}')
else:
    print(f'Dataset file missing at {data_file}')

### 2. Fit and Evaluate Estimator Candidates

In [ ]:
if 'X_train' in locals():
    candidate_models = {
        'Random Forest': build_pipeline('random_forest'),
        'XGBoost': build_pipeline('xgboost')
    }
    results = {}
    for name, pipeline in candidate_models.items():
        pipeline.fit(X_train, y_train)
        metrics = evaluate_model(pipeline, X_test, y_test)
        results[name] = metrics
        print(f'{name} -> R2: {metrics["r2"]}, MAE: {metrics["mae"]}, RMSE: {metrics["rmse"]}')
    display(pd.DataFrame(results).T)

### 3. Export Champion Model Artifact

In [ ]:
if 'candidate_models' in locals():
    best_model = candidate_models['XGBoost']
    os.makedirs('../models', exist_ok=True)
    artifact_path = '../models/best_model.pkl'
    joblib.dump(best_model, artifact_path)
    print(f'Champion model serialized to {artifact_path}')